## 🎯 Learning Objectives
* Understand the critical need for robust evaluation frameworks for AI agents.
* Differentiate between various agent evaluation paradigms, including automated metrics, human-in-the-loop, simulation-based, and LLM-as-a-Judge.
* Implement a basic evaluation pipeline for an AI agent, incorporating both rule-based and LLM-as-a-Judge metrics.
* Analyze the trade-offs and practical considerations when choosing and applying different evaluation strategies.
* Identify key performance indicators (KPIs) for agent evaluation, such as accuracy, latency, and token efficiency.


## Agent Evaluation Frameworks: Ensuring Reliability and Performance

Welcome to AG03-L11: Agent Evaluation Frameworks. As we delve deeper into building sophisticated AI agents, the question of *how well* they perform becomes paramount. Unlike traditional machine learning models, which often have clear, quantifiable metrics (e.g., accuracy on a classification task, RMSE on a regression task), evaluating AI agents presents unique challenges. Agents operate in dynamic environments, perform multi-step reasoning, utilize external tools, and often tackle open-ended problems where a single 'correct' answer might not exist.

### Why is Agent Evaluation Different and Difficult?

Imagine evaluating a junior employee versus a simple calculator. A calculator either gives the right answer or it doesn't – a straightforward check. An employee, however, might solve a problem in multiple ways, ask clarifying questions, use different tools, or even identify a better problem to solve. Evaluating the employee requires assessing their process, their resourcefulness, their communication, and the quality of their final output, often subjectively. AI agents are much more akin to the junior employee.

Key challenges include:

1.  **Non-Determinism**: LLMs, the core of many agents, are inherently probabilistic. The same prompt can yield different responses.
2.  **Multi-Step Reasoning**: Agents execute chains of thought and tool calls. Errors can propagate, and it's hard to pinpoint where a failure occurred.
3.  **Tool Use**: Evaluating tool efficacy, correct tool selection, and proper argument formatting adds complexity.
4.  **Open-Ended Tasks**: Many agent tasks (e.g., creative writing, complex problem-solving) lack a single ground truth.
5.  **Cost and Latency**: Running agents, especially with multiple LLM calls, can be expensive and slow, making extensive evaluation challenging.

### Core Paradigms for Agent Evaluation (2026 Perspective)

Modern agent evaluation frameworks combine several approaches to provide a holistic view of performance:

1.  **Automated Metrics (Rule-Based/Heuristic)**: These are the simplest and fastest. They involve checking for specific keywords, patterns, or structural elements in the agent's output. For example, checking if a math problem solver agent's output contains a numerical answer, or if a code generation agent's output is syntactically correct. While fast, they are often brittle and can miss nuanced errors.

2.  **Human-in-the-Loop (HITL) Evaluation**: The gold standard for subjective quality. Human evaluators assess the agent's output, reasoning steps, and overall helpfulness against predefined rubrics. This is crucial for tasks requiring creativity, common sense, or ethical considerations. Tools like Argilla and human annotation platforms facilitate this.

3.  **Simulation-Based Evaluation**: For agents interacting with environments (e.g., web browsing agents, game-playing agents), simulations provide a controlled, repeatable way to test robustness, exploration strategies, and goal achievement. This is common in robotics and reinforcement learning.

4.  **LLM-as-a-Judge Evaluation**: A rapidly evolving and powerful paradigm. Here, a separate, often more capable, LLM is prompted to act as an impartial judge. It receives the agent's prompt, its output, and sometimes the expected output or a set of criteria, then provides a score and a rationale. This offers a scalable way to get qualitative feedback, bridging the gap between automated metrics and expensive human evaluation. Frameworks like LangChain's `LangSmith` and LlamaIndex's `ResponseEvaluator` heavily leverage this.

### Key Performance Indicators (KPIs)

Beyond just correctness, we also care about:

*   **Accuracy/Correctness**: Did the agent achieve the desired outcome?
*   **Completeness**: Did it address all aspects of the prompt?
*   **Relevance**: Is the output pertinent to the query?
*   **Coherence/Fluency**: Is the output well-structured and easy to understand?
*   **Safety/Bias**: Does the output avoid harmful content or unfair biases?
*   **Latency**: How long did the agent take to complete the task?
*   **Cost (Token Usage)**: How many tokens (and thus API calls) did the agent consume?
*   **Tool Efficacy**: How effectively did the agent use its tools?

In this lesson, we'll focus on implementing a practical example combining automated metrics and the powerful LLM-as-a-Judge approach to evaluate a simple agent.


In [ ]:
import time
import re
from typing import List, Dict, Any, Tuple

# --- Mock Components for Demonstration ---

class MockLLM:
    """A mock LLM to simulate responses and token usage."""
    def __init__(self, model_name: str = "mock-gpt-4o-2026"): # Simulating a 2026-era LLM
        self.model_name = model_name
        self.call_count = 0

    def generate(self, prompt: str, temperature: float = 0.7) -> str:
        self.call_count += 1
        # Simulate LLM processing time and token usage
        time.sleep(0.1 + len(prompt) * 0.0005) # Longer prompts take more time
        
        # Simple mock logic based on prompt content
        if "calculate" in prompt.lower() or "solve" in prompt.lower():
            if "2+2" in prompt: return "The answer is 4."
            if "5*3" in prompt: return "The answer is 15."
            if "10/2" in prompt: return "The answer is 5."
            if "square root of 16" in prompt: return "The answer is 4."
            if "25-7" in prompt: return "The answer is 18."
            return "I need to use a calculator tool for this. The result is [CALC_RESULT]."
        
        if "judge the agent's response" in prompt.lower():
            # Mock judge logic: simple keyword matching for correctness
            if "correct" in prompt.lower() and "incorrect" not in prompt.lower():
                return "Score: 5/5. The agent's response was accurate and directly addressed the query. Reasoning: The final answer matches the expected outcome."
            elif "partially correct" in prompt.lower():
                return "Score: 3/5. The agent made some progress but the final answer was not fully correct. Reasoning: The agent missed a step or made a minor calculation error."
            else:
                return "Score: 1/5. The agent's response was incorrect or irrelevant. Reasoning: The agent failed to provide the correct answer."
        
        return f"Mock LLM response to: {prompt[:50]}..."

    def estimate_tokens(self, text: str) -> int:
        """Rough token estimation (e.g., 1 token per 4 characters)."""
        return len(text) // 4 + 1

class CalculatorTool:
    """A mock calculator tool."""
    def run(self, expression: str) -> str:
        try:
            # Basic and unsafe eval for demonstration. NEVER use eval with untrusted input.
            result = eval(expression)
            return str(result)
        except Exception as e:
            return f"Error calculating {expression}: {e}"

# --- Simple Agent Implementation ---

class SimpleMathAgent:
    """A simple agent that uses an LLM and a calculator tool to solve math problems."""
    def __init__(self, llm: MockLLM, calculator: CalculatorTool):
        self.llm = llm
        self.calculator = calculator
        self.tool_calls = []
        self.llm_calls = []

    def run(self, problem: str) -> Dict[str, Any]:
        self.tool_calls = []
        self.llm_calls = []
        start_time = time.time()
        
        # Initial thought process by LLM
        initial_prompt = f"You are a math assistant. Solve the following problem: {problem}. If you need to calculate, state 'CALCULATE: [expression]'. Otherwise, provide the final answer."
        llm_response = self.llm.generate(initial_prompt)
        self.llm_calls.append({'prompt': initial_prompt, 'response': llm_response})

        # Check if the LLM decided to use the calculator tool
        if "CALCULATE:" in llm_response:
            expression_match = re.search(r"CALCULATE: (.*)", llm_response)
            if expression_match:
                expression = expression_match.group(1).strip()
                self.tool_calls.append({'tool': 'calculator', 'input': expression})
                calc_result = self.calculator.run(expression)
                
                # Second LLM call to integrate tool result
                integration_prompt = f"I calculated '{expression}' and got '{calc_result}'. Now, based on the original problem '{problem}', provide the final answer using this result."
                final_answer = self.llm.generate(integration_prompt)
                self.llm_calls.append({'prompt': integration_prompt, 'response': final_answer})
            else:
                final_answer = "Agent failed to parse calculator expression."
        else:
            final_answer = llm_response # LLM directly provided an answer

        end_time = time.time()
        
        total_llm_tokens = sum(self.llm.estimate_tokens(call['prompt']) + self.llm.estimate_tokens(call['response']) for call in self.llm_calls)

        return {
            "final_answer": final_answer,
            "latency": end_time - start_time,
            "llm_calls_count": len(self.llm_calls),
            "tool_calls_count": len(self.tool_calls),
            "total_llm_tokens": total_llm_tokens,
            "llm_responses": [call['response'] for call in self.llm_calls]
        }

# --- Evaluation Framework --- 

def evaluate_agent_response(
    agent_output: Dict[str, Any],
    expected_answer: str,
    evaluation_criteria: str,
    llm_judge: MockLLM
) -> Dict[str, Any]:
    """Evaluates an agent's response using automated checks and an LLM-as-a-Judge."""
    
    evaluation_results = {
        "automated_correctness": False,
        "llm_judge_score": None,
        "llm_judge_reasoning": None,
        "metrics": {
            "latency": agent_output["latency"],
            "llm_calls": agent_output["llm_calls_count"],
            "tool_calls": agent_output["tool_calls_count"],
            "total_llm_tokens": agent_output["total_llm_tokens"]
        }
    }

    # 1. Automated Correctness Check (Rule-based)
    # For math problems, we can try to extract numbers and compare
    agent_final_answer_text = agent_output["final_answer"].lower()
    expected_answer_text = expected_answer.lower()

    if expected_answer_text in agent_final_answer_text:
        evaluation_results["automated_correctness"] = True
    else:
        # More robust check for numerical answers
        agent_numbers = re.findall(r'\d+', agent_final_answer_text)
        expected_numbers = re.findall(r'\d+', expected_answer_text)
        if agent_numbers and expected_numbers and agent_numbers[-1] == expected_numbers[-1]:
             evaluation_results["automated_correctness"] = True

    # 2. LLM-as-a-Judge Evaluation
    judge_prompt = f"""You are an expert AI agent evaluator. Your task is to judge the quality of an AI agent's response to a user query.

User Query: {evaluation_criteria}
Agent's Final Response: {agent_output['final_answer']}
Expected Answer: {expected_answer}

Based on the above, provide a score from 1 to 5 (5 being excellent, 1 being poor) and a brief reasoning. Focus on correctness, completeness, and relevance.

Format your response as: 'Score: [X/5]. Reasoning: [Your reasoning].'
"""
    
    judge_response = llm_judge.generate(judge_prompt, temperature=0.1) # Lower temp for more deterministic judging
    
    score_match = re.search(r"Score: (\d)/5", judge_response)
    reasoning_match = re.search(r"Reasoning: (.*)", judge_response)

    if score_match:
        evaluation_results["llm_judge_score"] = int(score_match.group(1))
    if reasoning_match:
        evaluation_results["llm_judge_reasoning"] = reasoning_match.group(1).strip()
    
    return evaluation_results

# --- Main Evaluation Run ---

if __name__ == "__main__":
    # Initialize components
    mock_llm = MockLLM()
    calculator_tool = CalculatorTool()
    agent = SimpleMathAgent(llm=mock_llm, calculator=calculator_tool)
    llm_judge = MockLLM(model_name="mock-judge-gpt-4o-2026") # A separate LLM for judging

    # Define a small evaluation dataset
    evaluation_dataset = [
        {
            "problem": "What is 2 plus 2?",
            "expected_answer": "The answer is 4.",
            "criteria": "The agent should correctly identify the sum of 2 and 2."
        },
        {
            "problem": "Calculate 5 multiplied by 3.",
            "expected_answer": "The answer is 15.",
            "criteria": "The agent should correctly calculate the product of 5 and 3."
        },
        {
            "problem": "What is 25 minus 7?",
            "expected_answer": "The answer is 18.",
            "criteria": "The agent should correctly calculate the difference between 25 and 7."
        },
        {
            "problem": "What is the square root of 16?",
            "expected_answer": "The answer is 4.",
            "criteria": "The agent should correctly find the square root of 16."
        },
        {
            "problem": "What is 10 divided by 3?", # Agent might struggle with non-integer division without specific instruction
            "expected_answer": "The answer is 3.33", # Expecting a rounded answer for this mock
            "criteria": "The agent should correctly calculate 10 divided by 3, potentially using the calculator tool."
        }
    ]

    print("--- Starting Agent Evaluation ---")
    all_evaluation_results = []

    for i, test_case in enumerate(evaluation_dataset):
        print(f"\n--- Test Case {i+1}: {test_case['problem']} ---")
        
        # Run the agent
        agent_output = agent.run(test_case["problem"])
        print(f"Agent's Final Answer: {agent_output['final_answer']}")
        print(f"Expected Answer: {test_case['expected_answer']}")

        # Evaluate the agent's response
        results = evaluate_agent_response(
            agent_output,
            test_case["expected_answer"],
            test_case["criteria"],
            llm_judge
        )
        all_evaluation_results.append(results)

        print(f"Automated Correctness: {results['automated_correctness']}")
        print(f"LLM Judge Score: {results['llm_judge_score']}/5")
        print(f"LLM Judge Reasoning: {results['llm_judge_reasoning']}")
        print(f"Latency: {results['metrics']['latency']:.2f}s")
        print(f"LLM Calls: {results['metrics']['llm_calls']}")
        print(f"Tool Calls: {results['metrics']['tool_calls']}")
        print(f"Total LLM Tokens: {results['metrics']['total_llm_tokens']}")

    print("\n--- Evaluation Summary ---")
    total_tests = len(all_evaluation_results)
    automated_correct_count = sum(1 for r in all_evaluation_results if r["automated_correctness"])
    avg_llm_judge_score = sum(r["llm_judge_score"] for r in all_evaluation_results if r["llm_judge_score"] is not None) / total_tests
    avg_latency = sum(r["metrics"]["latency"] for r in all_evaluation_results) / total_tests
    avg_tokens = sum(r["metrics"]["total_llm_tokens"] for r in all_evaluation_results) / total_tests

    print(f"Total Test Cases: {total_tests}")
    print(f"Automated Correctness Rate: {automated_correct_count / total_tests:.2%}")
    print(f"Average LLM Judge Score: {avg_llm_judge_score:.2f}/5")
    print(f"Average Latency per Test: {avg_latency:.2f}s")
    print(f"Average LLM Tokens per Test: {avg_tokens:.0f}")

    print("\n--- End of Evaluation ---")


### Interpreting the Output and Practical Considerations

The code above demonstrates a basic, yet illustrative, agent evaluation pipeline. Let's break down how to interpret its output and discuss its implications for real-world agent development.

#### Interpreting the Output

For each test case, you'll see:

*   **Agent's Final Answer vs. Expected Answer**: This is the direct comparison. Our mock agent's `MockLLM` is designed to give specific answers for certain problems, and a generic tool-use response for others. The `CalculatorTool` then provides the actual calculation.
*   **Automated Correctness**: This is a binary (True/False) check based on simple string matching or numerical extraction. It's fast and objective but can be brittle. For instance, if the expected answer is "4" and the agent says "The result is four.", the automated check might fail unless sophisticated NLP techniques are used. In our example, we use a simple regex to extract numbers, which is a step towards robustness.
*   **LLM Judge Score and Reasoning**: This is where the qualitative assessment comes in. The `MockLLM` acting as a judge provides a score (1-5) and a rationale. A score of 5 indicates excellent performance, while lower scores highlight areas for improvement. The reasoning is crucial for debugging and understanding *why* an agent failed or succeeded. Notice how the judge's reasoning can be more nuanced than a simple True/False.
*   **Metrics (Latency, LLM Calls, Tool Calls, Total LLM Tokens)**: These are quantitative performance indicators. They help you understand the efficiency and cost of your agent. A high number of LLM calls or tokens for a simple task might indicate an inefficient prompting strategy or an overly verbose agent. High latency directly impacts user experience.

Finally, the **Evaluation Summary** provides aggregate statistics, giving you an overall picture of your agent's performance across the dataset.

#### Performance Trade-offs

*   **Automated Checks**: 
    *   **Pros**: Extremely fast, cheap, deterministic, good for catching obvious regressions.
    *   **Cons**: Lacks nuance, easily fooled by superficial correctness, hard to scale for complex or open-ended tasks.
*   **LLM-as-a-Judge**: 
    *   **Pros**: Scalable qualitative feedback, can handle nuance, provides reasoning, faster and cheaper than human evaluation at scale.
    *   **Cons**: Can be biased by the judge LLM's own capabilities and biases, non-deterministic (though temperature can be lowered), still incurs LLM costs, prompt engineering for the judge is critical.
*   **Human-in-the-Loop**: 
    *   **Pros**: Gold standard for accuracy and subjective quality, handles extreme nuance, identifies novel failure modes.
    *   **Cons**: Very slow, very expensive, subjective (requires clear rubrics and multiple annotators for consistency).

#### Typical Use Cases

1.  **Regression Testing**: After making changes to your agent's prompt, tools, or architecture, run your evaluation suite to ensure no existing functionality has broken.
2.  **Comparative Analysis**: Evaluate different agent designs, prompting strategies, or LLM models against each other to determine the most effective approach.
3.  **Performance Monitoring**: In production, continuously evaluate a sample of agent interactions to detect performance degradation or new failure patterns.
4.  **Iterative Improvement**: Use the detailed feedback (especially from LLM-as-a-Judge reasoning) to refine agent prompts, add new tools, or adjust reasoning steps.
5.  **Benchmarking**: Establish a baseline performance for your agent and track progress over time as you iterate.

### Moving Forward

This example uses mock components, but in a real-world scenario, you would integrate with actual LLM APIs (e.g., OpenAI, Anthropic, Google Gemini) and robust tool implementations. Frameworks like LangChain's `LangSmith` and LlamaIndex provide more sophisticated and integrated evaluation capabilities, including dataset management, automated metric calculation, and LLM-as-a-Judge orchestration. The principles, however, remain the same: define clear objectives, choose appropriate metrics, and iterate based on feedback.


### Resources for Agent Evaluation Frameworks

To deepen your understanding and explore more advanced evaluation techniques, consider the following resources:

*   **LangChain Evaluation Documentation (LangSmith)**: The official documentation for LangChain's comprehensive evaluation platform, LangSmith, which offers robust tools for tracing, testing, and monitoring agents.
    *   [https://docs.smith.langchain.com/](https://docs.smith.langchain.com/)
    *   [https://python.langchain.com/docs/guides/evaluation/](https://python.langchain.com/docs/guides/evaluation/)

*   **LlamaIndex Evaluation Modules**: LlamaIndex also provides powerful evaluation capabilities, particularly for RAG (Retrieval Augmented Generation) applications, which are often at the core of many agents.
    *   [https://docs.llamaindex.ai/en/stable/module_guides/evaluating/root.html](https://docs.llamaindex.ai/en/stable/module_guides/evaluating/root.html)

*   **Ragas: Evaluation Framework for RAG**: While focused on RAG, Ragas introduces concepts and metrics highly relevant to evaluating agents that retrieve information.
    *   [https://docs.ragas.io/en/latest/](https://docs.ragas.io/en/latest/)

*   **Google AI Studio / Gemini API Documentation**: For integrating Google's latest LLMs into your agents and evaluation pipelines.
    *   [https://ai.google.dev/](https://ai.google.dev/)

*   **OpenAI API Documentation**: For integrating OpenAI's models (like GPT-4o) into your agents and evaluation pipelines.
    *   [https://platform.openai.com/docs/](https://platform.openai.com/docs/)

*   **Hugging Face Evaluate Library**: A general-purpose library for ML model evaluation, which can be adapted for certain aspects of agent evaluation.
    *   [https://huggingface.co/docs/evaluate/index](https://huggingface.co/docs/evaluate/index)

*   **Research Papers on Agent Evaluation**: Search for recent papers on arXiv or major AI conferences (NeurIPS, ICLR, ACL) focusing on "LLM agent evaluation," "tool-use evaluation," or "multi-step reasoning evaluation" for the latest academic insights.
